# machine-learning_012

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (233).ipynb`

| Field | Value |
|---|---|
| Section | `machine_learning` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 8 |
| Detected functions | adaptive_intraday_momentum_signal, label_signals, generate_features, predict_signals, align_signals_to_ohlcv, run_pipeline |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

In [ ]:
ohlcv_train = pd.read_csv("/content/drive/MyDrive/data_train.csv")
ohlcv_train

In [ ]:
ohlcv_test = pd.read_csv("/content/drive/MyDrive/5min_data.csv")
ohlcv_test

# Convert 'Date' column to datetime if it isn't already
ohlcv_test['Date'] = pd.to_datetime(ohlcv_test['Date'])

# Define the date range
start_date = '2024-01-01'
end_date = '2024-12-30'

# Define the tickers you want to filter
selected_tickers = ['ADANIENT', 'ADANIPORTS', 'APOLLOHOSP', 'ASIANPAINT', 'AXISBANK',
       'BAJAJ-AUTO', 'BAJAJFINSV', 'BAJFINANCE', 'BEL', 'BHARTIARTL',
       'BPCL', 'BRITANNIA', 'CIPLA', 'COALINDIA', 'DRREDDY', 'EICHERMOT',
       'GRASIM', 'HCLTECH', 'HEROMOTOCO', 'HINDALCO', 'HINDUNILVR',
       'ICICIBANK', 'INDUSINDBK', 'INFY', 'ITC', 'JSWSTEEL', 'KOTAKBANK',
       'LT', 'MARUTI', 'MM', 'NESTLEIND', 'NTPC', 'ONGC', 'POWERGRID',
       'RELIANCE', 'SBILIFE', 'SBIN', 'SHRIRAMFIN', 'SUNPHARMA',
       'TATACONSUM', 'TATAMOTORS', 'TATASTEEL', 'TCS', 'TECHM', 'TITAN',
       'TRENT', 'ULTRACEMCO', 'WIPRO']  # Add tickers you wa

# Apply filtering
ohlcv_test = ohlcv_test[
    (ohlcv_test["Date"] >= start_date) &
    (ohlcv_test["Date"] <= end_date) &
    (ohlcv_test["Ticker"].isin(selected_tickers))
]

# Display filtered results
ohlcv_test

In [ ]:
tradebook = pd.read_csv('/content/drive/MyDrive/tradebook.csv')
tradebook

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier

# === Signal Function ===
def adaptive_intraday_momentum_signal(df, base_window=15, min_window=5,
                                      vol_spike_factor=1.5, vol_increase_thresh=0.3,
                                      momentum_thresh=0.001, vol_ewma_span=3,
                                      vol_filter_quantile=0.8, vol_filter_window=30):
    df = df.copy()
    df['Vol_smooth'] = df['volume'].ewm(span=vol_ewma_span, adjust=False).mean()
    df['Vol_avg_30'] = df['Vol_smooth'].rolling(window=30, min_periods=1).mean()
    df['Vol_sum_5'] = df['Vol_smooth'].rolling(window=5, min_periods=1).sum()

    vol_spike_mask = df['Vol_sum_5'] > vol_spike_factor * df['Vol_avg_30']
    momentum_window = np.where(vol_spike_mask, min_window, base_window).astype(int)

    df['LogReturn'] = np.log(df['close']).diff()
    df['Volatility'] = df['LogReturn'].rolling(window=vol_filter_window, min_periods=1).std()
    vol_thresh = df['Volatility'].quantile(vol_filter_quantile)

    n = len(df)
    signals = np.zeros(n, dtype=int)

    close = df['close'].values
    vol_smooth = df['Vol_smooth'].values
    volatility = df['Volatility'].values

    for i in range(n):
        window = momentum_window[i]
        if i < window or volatility[i] > vol_thresh:
            continue

        price_now = close[i]
        price_past = close[i - window]
        vol_now = vol_smooth[i]
        vol_past = vol_smooth[i - window]

        if price_past <= 0 or vol_past <= 0:
            continue

        momentum = np.log(price_now / price_past)
        vol_momentum = (vol_now / vol_past) - 1

        if momentum > momentum_thresh and vol_momentum > vol_increase_thresh:
            signals[i] = 1
        elif momentum < -momentum_thresh and vol_momentum > vol_increase_thresh:
            signals[i] = 2

    df['Signal'] = signals
    df.drop(columns=['Vol_smooth', 'Vol_avg_30', 'Vol_sum_5', 'LogReturn', 'Volatility'], inplace=True)
    return df

# === Labeling Function ===
def label_signals(df, tradebook):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    tradebook = tradebook.copy()
    tradebook['Entry Time'] = pd.to_datetime(tradebook['Entry Time'])
    tradebook['Signal'] = tradebook['Direction'].map({'long': 1, 'short': 2})
    tradebook['label'] = (tradebook['Exit Reason'].fillna('') == 'Take Profit').astype(int)

    merge_cols = ['Ticker', 'Signal']
    merged = pd.merge(df[df['Signal'] != 0],
                      tradebook[['Entry Time', 'Ticker', 'Signal', 'label']],
                      left_on=['Date'] + merge_cols,
                      right_on=['Entry Time'] + merge_cols,
                      how='left')

    df.loc[merged.index, 'label'] = merged['label']
    df['label'] = df['label'].fillna(0).astype(int)
    return df[df['Signal'] != 0].copy()

# === Feature Engineering ===
def generate_features(df):
    df = df.copy()
    df['Return_1'] = df.groupby('Ticker')['close'].pct_change(1)
    df['Return_5'] = df.groupby('Ticker')['close'].pct_change(5)
    df['Volatility_5'] = df.groupby('Ticker')['close'].transform(lambda x: x.pct_change().rolling(5).std())
    df['Volume_Change'] = df.groupby('Ticker')['volume'].pct_change(1)
    df['Spread'] = (df['high'] - df['low']) / df['close']
    df['Book_Imbalance'] = df['Signal'].map({1: 0.5, 2: -0.5})  # Placeholder feature

    features = ['Return_1', 'Return_5', 'Volatility_5', 'Volume_Change', 'Spread', 'Book_Imbalance']
    df[features] = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
    return df, features

# === Predict ===
def predict_signals(model, df, features):
    X = df[features].astype(np.float32)
    df['Success_Prob'] = model.predict_proba(X)[:, 1]
    df['Filtered_Signal'] = np.where(df['Success_Prob'] > 0.005, df['Signal'], 0)
    return df

# === Align signals to full OHLCV bars ===
def align_signals_to_ohlcv(ohlcv_df, signals_df):
    ohlcv_df = ohlcv_df.copy()
    signals_df = signals_df.copy()
    ohlcv_df['Date'] = pd.to_datetime(ohlcv_df['Date'])
    signals_df['Date'] = pd.to_datetime(signals_df['Date'])

    signals_sub = signals_df[['Ticker', 'Date', 'Filtered_Signal']]

    merged = pd.merge(
        ohlcv_df,
        signals_sub,
        on=['Ticker', 'Date'],
        how='left'
    )

    merged['Filtered_Signal'] = merged['Filtered_Signal'].fillna(0).astype(int)
    return merged

# === Full Pipeline ===
def run_pipeline(ohlcv_train, ohlcv_test, tradebook):
    print("==> Generating signals on training data")
    ohlcv_train = adaptive_intraday_momentum_signal(ohlcv_train)

    print("==> Generating signals on test data")
    ohlcv_test = adaptive_intraday_momentum_signal(ohlcv_test)

    print("==> Labeling training signals")
    train_labeled = label_signals(ohlcv_train, tradebook)

    print("==> Feature engineering on training data")
    train_feat, features = generate_features(train_labeled)

    print("==> Training model")
    X = train_feat[features].astype(np.float32)
    y = train_feat['label'].fillna(0).replace([np.inf, -np.inf], 0).astype(int)
    model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
    model.fit(X, y)

    print("==> Feature engineering on test data")
    test_feat = ohlcv_test.copy()  # Keep full OHLCV for alignment later
    test_feat_signals = test_feat[test_feat['Signal'] != 0].copy()
    test_feat_signals, _ = generate_features(test_feat_signals)

    print("==> Predicting and filtering")
    predicted_signals = predict_signals(model, test_feat_signals, features)

    print("==> Aligning signals to full 5-min OHLCV bars")
    aligned_result = align_signals_to_ohlcv(test_feat, predicted_signals)

    return aligned_result

result_df = run_pipeline(ohlcv_train, ohlcv_test, tradebook)

result_df

In [ ]:
result_df.to_csv("/content/drive/MyDrive/result.csv")
result_df

In [ ]:
result_df['Filtered_Signal'].unique()